In [1]:
#!pip install xgboost joblib scikit-learn pandas numpy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

In [2]:
np.random.seed(42)
n_muestras = 1000

edad = np.random.randint(40, 85, n_muestras)
sistolica = np.random.randint(100, 180, n_muestras)
diastolica = np.random.randint(60, 110, n_muestras)
toma_medicamento = np.random.choice([0, 1], size=n_muestras, p=[0.3, 0.7]) # 70% sí se lo toma

In [3]:
# Crear la regla médica para la columna objetivo (Crisis: 1 = Sí, 0 = No)
# Riesgo si la presión es alta y NO tomó medicamento, o si la presión es críticamente alta
riesgo_score = (sistolica * 0.5) + (diastolica * 0.3) - (toma_medicamento * 30)
crisis = np.where(riesgo_score > 65, 1, 0)

In [4]:
# Crear el DataFrame (la tabla de datos)
df = pd.DataFrame({
    'Edad': edad,
    'Sistolica': sistolica,
    'Diastolica': diastolica,
    'Toma_Medicamento': toma_medicamento,
    'Crisis': crisis
})

In [5]:
# 2. DIVIDIR DATOS (Características X, Objetivo y)
X = df.drop('Crisis', axis=1)
y = df['Crisis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# 3. DEFINICIÓN DE LOS 3 ALGORITMOS 
modelos = {
    'Regresión Logística': LogisticRegression(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42)
}

In [7]:
# 4. ENTRENAR Y EVALUAR
resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train) # Entrenamiento
    predicciones = modelo.predict(X_test) # Predicción
    
    # Calcular métricas
    resultados.append({
        'Algoritmo': nombre,
        'Exactitud (Accuracy)': accuracy_score(y_test, predicciones),
        'Precisión (Precision)': precision_score(y_test, predicciones),
        'Recall': recall_score(y_test, predicciones),
        'F1-Score': f1_score(y_test, predicciones)
    })

In [8]:
# Mostrar la tabla comparativa de métricas
df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))

          Algoritmo  Exactitud (Accuracy)  Precisión (Precision)   Recall  F1-Score
Regresión Logística                 0.975               0.969466 0.992188  0.980695
      Random Forest                 0.975               0.992000 0.968750  0.980237
            XGBoost                 0.980               0.992063 0.976562  0.984252


In [9]:
# 5. GUARDAR EL GANADOR (XGBoost) para la API
joblib.dump(modelos['XGBoost'], 'mejor_modelo_htas.pkl')
print("\n¡Modelo XGBoost guardado exitosamente como 'mejor_modelo_htas.pkl'!")


¡Modelo XGBoost guardado exitosamente como 'mejor_modelo_htas.pkl'!


In [ ]:
#Creación del API
#De momento se meterá aquí, en esta celda será la importación de las librerías
#Descarga de la librería de FastAPI
!pip install fastapi uvicorn nest-asyncio pydantic
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import uvicorn
import nest_asyncio

In [ ]:
# CORDENADA COMPLETA EN UNA SOLA CELDA
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import uvicorn
import nest_asyncio

# Permitir ejecuciones asíncronas en Jupyter
nest_asyncio.apply()

app = FastAPI(title="API de Monitoreo HTAS")
modelo = joblib.load('mejor_modelo_htas.pkl')

class DatosPaciente(BaseModel):
    Edad: int
    Sistolica: int
    Diastolica: int
    Toma_Medicamento: int

@app.post("/predecir_crisis")
def predecir_crisis(datos: DatosPaciente):
    datos_df = [[datos.Edad, datos.Sistolica, datos.Diastolica, datos.Toma_Medicamento]]
    prediccion = int(modelo.predict(datos_df)[0])
    probabilidad = float(modelo.predict_proba(datos_df)[0][1])
    
    alerta = "ALTA: Riesgo de crisis." if prediccion == 1 else "NORMAL: Estable."
    return {"crisis_detectada": prediccion, "probabilidad_riesgo": f"{probabilidad * 100:.2f}%", "alerta_clinica": alerta}

if __name__ == "__main__":
    print("🚀 API en línea. Esperando datos del baumanómetro...")
    
    # ESTE ES EL CAMBIO PARA JUPYTER:
    import asyncio
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    
    # En lugar de uvicorn.run(), usamos el bucle que ya existe en Jupyter:
    await server.serve()

INFO:     Started server process [37480]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


🚀 API en línea. Esperando datos del baumanómetro...
INFO:     127.0.0.1:62021 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:62021 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:62046 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:62046 - "GET /openapi.json HTTP/1.1" 200 OK
